# 01 · Data Quality Audit and Understanding

In [1]:
import json
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

from credit_risk.config import PROJECT_ROOT, RAW_DATA_DIR, REPORTS_DIR
from credit_risk.data_utils import load_csv
from credit_risk.data_quality import audit_dataframe, AuditConfig

raw_path = RAW_DATA_DIR / "application_train.csv"
if not raw_path.exists():
    raise FileNotFoundError(
        f"Missing {raw_path}. Place Kaggle's application_train.csv in data/raw/."
    )

df = load_csv(raw_path)
print(f"Source dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

Source dataset: 307,511 rows × 122 columns


## 0.1 ·  `ID` and `TARGET` Check

In [2]:
source_checks = pd.DataFrame([
    {"Check": "Rows", "Result": len(df)},
    {"Check": "Columns", "Result": df.shape[1]},
    {"Check": "Unique SK_ID_CURR", "Result": df["SK_ID_CURR"].nunique(dropna=True)},
    {"Check": "Missing SK_ID_CURR", "Result": int(df["SK_ID_CURR"].isna().sum())},
    {"Check": "Missing TARGET", "Result": int(df["TARGET"].isna().sum())},
    {"Check": "TARGET categories", "Result": sorted(df["TARGET"].dropna().unique().tolist())},
])
display(source_checks)

if df["TARGET"].isna().any() or not df["TARGET"].isin([0, 1]).all():
    raise ValueError("TARGET is not a complete binary label; inspect it before splitting.")

,Check,Result
0,Rows,307511
1,Columns,122
2,Unique SK_ID_CURR,307511
3,Missing SK_ID_CURR,0
4,Missing TARGET,0
5,TARGET categories,"[0, 1]"


## 0.2 [Data dictionary](docs\HomeCredit_columns_description.md)

In [16]:
dictionary_path = PROJECT_ROOT / "docs" / "home_credit_application_variable_types.csv"
metadata = pd.read_csv(dictionary_path)
reference = metadata.drop_duplicates("Row").set_index("Row")

comparison = pd.DataFrame({
    "pandas_dtype": df.dtypes.astype(str)
}).join(reference[["Description", "Statistical Type"]], how="left")

print(f"Metadata coverage: {comparison['Statistical Type'].notna().sum()} / {len(df.columns)} variables")

display(pd.crosstab(
    comparison["Statistical Type"],
    comparison["pandas_dtype"],
    margins=True
))

Metadata coverage: 122 / 122 variables


pandas_dtype,float64,int64,object,All
Statistical Type,,,,
Binary nominal categorical,0,32,2,34
Binary nominal categorical (target),0,1,0,1
Continuous quantitative (monetary),4,0,0,4
Continuous quantitative (normalized),47,0,0,47
Cyclical nominal categorical (day of week),0,0,1,1
Discrete cyclical time (hour of day),0,1,0,1
Discrete quantitative (count),12,1,0,13
Identifier (nominal key),0,1,0,1
Nominal categorical,0,0,12,12


## 0.3 · Train (80%) / Test (20%)

In [4]:
train_df, test_df = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df["TARGET"]
)
X_train = train_df.drop(columns=["SK_ID_CURR", "TARGET"])
print("Training:", train_df.shape, " | Holdout:", test_df.shape)
print("Predictors audited:", X_train.shape[1])

Training: (246008, 122)  | Holdout: (61503, 122)
Predictors audited: 120


In [17]:
pd.DataFrame({
    "Original": df["TARGET"].value_counts(normalize=True),
    "Train": train_df["TARGET"].value_counts(normalize=True),
    "Test": test_df["TARGET"].value_counts(normalize=True)
}).mul(100).round(2)

,Original,Train,Test
TARGET,,,
0,91.93,91.93,91.93
1,8.07,8.07,8.07


In [5]:
audit = audit_dataframe(
    train_df,
    metadata=metadata,
    special_values={
        "DAYS_EMPLOYED": [365243],
        "CODE_GENDER": ["XNA"],
        "ORGANIZATION_TYPE": ["XNA"],
    },
    config=AuditConfig(),
)

report_paths = audit.export(REPORTS_DIR / "data_quality")
print("HTML report:", report_paths["html"])
print("Structured JSON:", report_paths["json"])
print("Open the HTML file in a web browser to navigate the six audit sections.")

HTML report: C:\Universidad\Semestre 7\home-credit-risk-ml\reports\data_quality\audit_report.html
Structured JSON: C:\Universidad\Semestre 7\home-credit-risk-ml\reports\data_quality\audit_findings.json
Open the HTML file in a web browser to navigate the six audit sections.


## Quality Analisis 

### 1. Understading of the data

1.1 **Estructure**: `application_train.csv` contains 307,511 loan applications, 122 variables from wich 120 are predictors, `SK_ID_CURR` is a identifier and `TARGET` is the respond variable

1.2 **`TARGET`**: Indicates if the client had dificulties or not with payment, only 8.07% belongs to the 1 class, *we are working with important class imbalance*

1.3 **Test/Train**: We hold 61,503 observations for evaluating future models, because we are working with imbalace classes we used estratificated splitting

### 2. Variable Types

2.1 Many variables are wrongly represented, we need to correct this in the next notebook

### 3. Missing Values, Especial Values and Incosistencies

3.1. 67 covariables present missing values, in wich 41 has more than 50% of their observations incomplete, furthermore, 97.23% of the observations has at least 1 missing value, therefore dropping na values is not an opttion

3.2. **Paterns**: We identified 16 groups with identical missing data across 52 variables

3.3 **Especial categories**: XNA

3.4 **Constant variables**: 4

3.5 **Integrity**: duplicated rows, equal columns

## Cleaning Plan

1. Verify duplicates, constants and incompatibilities

2. Define a treatment for especial values

3. Missing Values: Identify diferences between real missing values from patterns of missing values, evaluate imputation or create an missing value indicator